In [2]:
from __future__ import annotations

import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [3]:
df = pd.read_stata("../data/scf_processed.dta")
df.head()

,wgt,hhsex,age,agecl,edcl,married,lf,racecl,racecl4,racecl5,...,racecl4_lbl,racecl5_lbl,race_lbl,age_lbl,equityfin,finincome,finmill,incomemill,incomethou,findeciles
0,6859.959728,2,35,2,2,2,1,1,1,1,...,white non-Hispanic,white non-Hispanic,white non-Hispanic,(30-35],NaN,0.0,0.0,0.012026,12.026084,0.0
1,7218.403805,2,35,2,2,2,1,1,1,1,...,white non-Hispanic,white non-Hispanic,white non-Hispanic,(30-35],NaN,0.0,0.0,0.012026,12.026084,0.0
2,6982.387611,2,35,2,2,2,1,1,1,1,...,white non-Hispanic,white non-Hispanic,white non-Hispanic,(30-35],NaN,0.0,0.0,0.012026,12.026084,0.0
3,7057.994966,2,35,2,2,2,1,1,1,1,...,white non-Hispanic,white non-Hispanic,white non-Hispanic,(30-35],NaN,0.0,0.0,0.012026,12.026084,0.0
4,6934.706832,2,35,2,2,2,1,1,1,1,...,white non-Hispanic,white non-Hispanic,white non-Hispanic,(30-35],NaN,0.0,0.0,0.012026,12.026084,0.0


In [4]:
filtered_df = df[
    (df["racecl4_lbl"] != "Other or Multiple race")
    & df["age"].between(21, 80)
    & (df["networth"] > 0)
    & (df["asset"] > 0)
    & (df["fin"] > 0)
    & (df["income"])
    & (df["equityfin"]).between(0, 1)
]

In [5]:
filtered_df.columns

Index(['wgt', 'hhsex', 'age', 'agecl', 'edcl', 'married', 'lf', 'racecl',
       'racecl4', 'racecl5', 'race', 'racecl_ex', 'income', 'wageinc',
       'yesfinrisk', 'nofinrisk', 'hborrfin', 'hsavfin', 'hsavnfin', 'finlit',
       'othfin', 'hothfin', 'equity', 'hequity', 'fin', 'hfin', 'othnfin',
       'hothnfin', 'nfin', 'hnfin', 'nhnfin', 'asset', 'hasset', 'networth',
       'assetcat', 'year', 'refin_ever', 'bfinpro', 'bself', 'ifinpro',
       'iself', 'bfinplan', 'ifinplan', 'hhsex_lbl', 'edcl_lbl', 'married_lbl',
       'lf_lbl', 'racecl_lbl', 'racecl4_lbl', 'racecl5_lbl', 'race_lbl',
       'age_lbl', 'equityfin', 'finincome', 'finmill', 'incomemill',
       'incomethou', 'findeciles'],
      dtype='object')

In [6]:
glm = smf.glm(
    "hequity ~ hhsex_lbl + age + edcl_lbl + married_lbl + lf_lbl + racecl4_lbl + year"
    "+ finmill  + incomemill",
    data=filtered_df,
    freq_weights=filtered_df["wgt"],
    family=sm.families.Binomial(),
)

res = glm.fit()

print(res.summary2())

                                    Results: Generalized linear model
Model:                           GLM                         AIC:                       1004114410.3032  
Link Function:                   Logit                       BIC:                       -19960157332.2624
Dependent Variable:              hequity                     Log-Likelihood:            -5.0206e+08      
Date:                            2024-03-24 19:56            LL-Null:                   -6.9540e+08      
No. Observations:                225424                      Deviance:                  1.0041e+09       
Df Model:                        22                          Pearson chi2:              5.13e+20         
Df Residuals:                    1011091543                  Scale:                     1.0000           
Method:                          IRLS                                                                    
----------------------------------------------------------------------------------

In [7]:
glm = smf.glm(
    "hequity ~ hhsex_lbl + age + edcl_lbl + married_lbl + lf_lbl + racecl4_lbl + year"
    "+ finincome",
    data=filtered_df,
    family=sm.families.Binomial(),
    freq_weights=filtered_df["wgt"],
)
res = glm.fit()
print(res.summary2())

                                    Results: Generalized linear model
Model:                           GLM                         AIC:                       1113391144.9558  
Link Function:                   Logit                       BIC:                       -19850880616.3440
Dependent Variable:              hequity                     Log-Likelihood:            -5.5670e+08      
Date:                            2024-03-24 19:56            LL-Null:                   -6.9540e+08      
No. Observations:                225424                      Deviance:                  1.1134e+09       
Df Model:                        21                          Pearson chi2:              2.81e+20         
Df Residuals:                    1011091544                  Scale:                     1.0000           
Method:                          IRLS                                                                    
----------------------------------------------------------------------------------

In [8]:
conditional_df = filtered_df[filtered_df["hequity"] > 0]

In [9]:
glm = smf.glm(
    "equityfin ~ hhsex_lbl + age + edcl_lbl + married_lbl + lf_lbl + racecl4_lbl + year"
    "+ finmill  + incomemill",
    data=conditional_df,
    family=sm.families.Binomial(),
    freq_weights=conditional_df["wgt"],
)
res = glm.fit()
print(res.summary2())

                                    Results: Generalized linear model
Model:                           GLM                         AIC:                       581145689.9561   
Link Function:                   Logit                       BIC:                       -11028157031.6221
Dependent Variable:              equityfin                   Log-Likelihood:            -2.9057e+08      
Date:                            2024-03-24 19:56            LL-Null:                   -2.9451e+08      
No. Observations:                146338                      Deviance:                  2.0799e+08       
Df Model:                        22                          Pearson chi2:              5.10e+16         
Df Residuals:                    557910058                   Scale:                     1.0000           
Method:                          IRLS                                                                    
----------------------------------------------------------------------------------

In [10]:
glm = smf.glm(
    "equityfin ~ hhsex_lbl + age + edcl_lbl + married_lbl + lf_lbl + racecl4_lbl + year"
    "+ finincome",
    data=conditional_df,
    family=sm.families.Binomial(),
    freq_weights=conditional_df["wgt"],
)
res = glm.fit()
print(res.summary2())

                                    Results: Generalized linear model
Model:                           GLM                         AIC:                       581888722.1621   
Link Function:                   Logit                       BIC:                       -11027414017.5558
Dependent Variable:              equityfin                   Log-Likelihood:            -2.9094e+08      
Date:                            2024-03-24 19:56            LL-Null:                   -2.9451e+08      
No. Observations:                146338                      Deviance:                  2.0873e+08       
Df Model:                        21                          Pearson chi2:              1.97e+08         
Df Residuals:                    557910059                   Scale:                     1.0000           
Method:                          IRLS                                                                    
----------------------------------------------------------------------------------